In [8]:
# Importations

import time
from enderscope import SerialUtils, Stage 
import serial
from math import *
import threading

In [ ]:
# Variables

  # Variables modifiables

Diametre = 40 # en mm
coordonees_initiales = [10,10] # coordonnées origine [gauche, bas] du carré lié à la position de la boîte de pétri
vitesse_extrusion = 30

  #  Variables fixes

epaisseur = 5 # en mm (largeur x épaisseur = V_seringue x Surface seringue / V_imprimante)
nombre_passage = ceil(Diametre /(2 * epaisseur)) # arrondit à l'entier superieur 
 
decalage = [1,1] # Pour le pousse seringue A, décalage de 1 selon x et 1 selon y. Si centré, on en deduit les decalages des autres 

hauteur_impression = 10 # en mm, dépend en partie de la hauteur de la boîte de pétri

In [10]:
# Ports

ports = SerialUtils.serial_ports() # Liste des ports
print (ports) # Affiche la liste

port_pousse_seringue = ports[0] # A modifier en fonction du branchement
port_imprimante = ports[1] # A modifier en fonction du branchement

s = Stage(port_imprimante, 115200) # Connexion imprimante
pousse_seringue = serial.Serial(port= port_pousse_seringue, baudrate=115200, timeout=0.01, writeTimeout=1) # Connexion pousse seringue

['COM6', 'COM7']


In [ ]:
# Message respectif à envoyer à un pousse seringue pour le débloquer

def message_depart(lettre_pousse_seringue):
    if lettre_pousse_seringue == "A" :  # Le '\n' correspond à la fin du message
        return b"A\n" 
    if lettre_pousse_seringue == "B":
        return b"B\n"
    if lettre_pousse_seringue == "C":
        return b"C\n"

In [ ]:
# Message respectif à envoyer à un pousse seringue pour l'arrêter 

def message_arret(lettre_pousse_seringue):
    if lettre_pousse_seringue == "A" :  # Le '\n' correspond à la fin du message
        return b"S\n" 
    if lettre_pousse_seringue == "B":
        return b"T\n"
    if lettre_pousse_seringue == "C":
        return b"U\n"


In [12]:
# Déterminer la postition qui'il faut demadnder à l'imprimante en fonction du pousse seringue utilisé

def position(lettre_pousse_seringue, coordonnees):
    position_tube = []
    if lettre_pousse_seringue == "A":
        position_tube.append(coordonnees[0] - decalage[0])
        position_tube.append(coordonnees[1] - decalage[1])
        return position_tube
    
    if lettre_pousse_seringue == "B":
        position_tube.append(coordonnees[0] + decalage[0])
        position_tube.append(coordonnees[1] - decalage[1])
        return position_tube

    if lettre_pousse_seringue == "C":
        position_tube.append(coordonnees[0] + decalage[0])
        position_tube.append(coordonnees[1] + decalage[1])
        return position_tube

In [ ]:
# Forme un carré

def carre(): # Position du coin (bas_gauche) du carre

    Diametre_carre = Diametre

    for i in range (nombre_passage):
        s.write_code(f"M203 X30")
        s.write_code(f"M203 Y30")
        s.move_axis('x', Diametre_carre)
        s.move_axis ('y', Diametre_carre)
        s.move_axis('x', -Diametre_carre)
        s.move_axis('y', - (Diametre_carre -epaisseur))

    

        s.move_axis('x', epaisseur)
        
        Diametre_carre = Diametre_carre - (2 * epaisseur)
    s.write_code(f"M400")

In [ ]:
# On fait le focus

s.home()

# Tracer le premier carré avec le pousse seringue A

s.move_absolute(position("A", coordonees_initiales)[0], position("A",coordonees_initiales)[1], hauteur_impression) # On se met a la position pour le pousse seringue en question et non l'origine
s.write_code(f"M400") # Permet d'attendre que le mouvement soit fini
pousse_seringue.write(message_depart('A'))
carre() 
pousse_seringue.write(message_arret('A'))

# Tracer le deuxième carré avec le pousse seringue B

coordonees_initiales_carre_2 = [coordonees_initiales[0]+Diametre, coordonees_initiales[1]] # Coordoones de l origine pour le carre 2
s.move_absolute(position("A", coordonees_initiales_carre_2)[0], position("A",coordonees_initiales_carre_2)[1], hauteur_impression)
s.write_code(f"M400") 
pousse_seringue.write(message_depart('A'))
carre() 
pousse_seringue.write(message_arret('A'))

# Tracer le troisième carré avec le pousse seringue C

coordonees_initiales_carre_3 = [coordonees_initiales[0]+ 2 *Diametre, coordonees_initiales[1]]
s.move_absolute(position("A", coordonees_initiales_carre_3)[0], position("A",coordonees_initiales_carre_3)[1], hauteur_impression)
s.write_code(f"M400") 
pousse_seringue.write(message_depart('A'))
carre() 
pousse_seringue.write(message_arret('A'))

2